# 🚀 Team Data Analysis Notebook

## Overview

This notebook performs **data ingestion, transformation, and analysis** on our team's dataset. The workflow includes:

1. *Loading raw data* from the source
2. **Cleaning and transforming** the records
3. Running *exploratory analysis* and generating insights
4. Exporting final results for reporting

---

## Author

**Name:** <span style="color: #2E86C1;">Arul</span> *(Data Engineer)*

---

## Our Team

![Team Photo](https://fpimages.withfloats.com/actual/6929d1ac956d0a744b5c9822.jpeg)

*The amazing team behind this project!*

---

### Notes
- **Maintained by:** Arul
- **Last updated:** _{{update with run date}_
- **Contact:** Reach out via email for questions

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS usage_metrics


In [0]:
# Load mobile OS usage data from the public API and write it into the volume

import requests

url = "https://public.tableau.com/app/sample-data/mobile_os_usage.csv"
response = requests.get(url)
response.raise_for_status()  # fail fast on HTTP errors

# Pull the data into a Python variable
data_text = response.text

print(f"Fetched {len(data_text)} bytes from {url}")
print(data_text[:500])  # preview the first few lines

# Write the data into the volume
volume_path = "/Volumes/workspace/default/usage_metrics/mobile_os_usage.csv"
dbutils.fs.put(volume_path, data_text, overwrite=True)

print(f"Data written to {volume_path}")

In [0]:
%fs ls /Volumes/workspace/default/usage_metrics

In [0]:
%fs head /Volumes/workspace/default/usage_metrics/mobile_os_usage.csv

In [0]:
%run ./4_child_nb_dataload

In [0]:
df1 = spark.read.csv(
    "/Volumes/workspace/default/usage_metrics/mobile_os_usage.csv",
    header=True,
    inferSchema=True,
)
df2=df1.toDF("date", "mobile_os", "usage_percent")
df2.display()

In [0]:
# Rename columns to snake_case (Delta rejects spaces in column names)
df1 = (
    df1.withColumnRenamed("Date", "date")
    .withColumnRenamed("Mobile Operating System", "mobile_os")
    .withColumnRenamed("Percent of Usage", "usage_percent")
)

# Write df1 into a Delta table in the default schema
df1.write.mode("overwrite").saveAsTable("default.mobile_os_usage")

print("Data written to table default.mobile_os_usage")

spark.table("default.mobile_os_usage").display()

In [0]:
spark.sql("SELECT * FROM default.mobile_os_usage").display()

In [0]:
def to_uppercase(value):
    """Convert the input to uppercase.

    Works on a single string, a list of strings, or a PySpark Column.
    Non-string values are converted to their string form first.
    """
    if isinstance(value, str):
        return value.upper()
    elif isinstance(value, list):
        return [str(v).upper() for v in value]
    else:
        return str(value).upper()


# Example usage
print(to_uppercase("android"))            # ANDROID
print(to_uppercase(["ios", "Bada", 42]))  # ['IOS', 'BADA', '42']

In [0]:
%pip install pandas

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
df=pd.read_csv("/Volumes/workspace/default/usage_metrics/mobile_os_usage.csv") 
df.display()

In [0]:
%sh
echo "Magic commands tasks completed"